In [1]:
from sqlalchemy import create_engine, text, inspect
from sqlalchemy.orm import sessionmaker
import pandas as pd
from unidecode import unidecode
import numpy as np


In [2]:
#from datetime import datetime
#CREACION DE LA CONEXION CON BASE DE DATOS SQL SERVER

# Configura los detalles de la conexión
host='172.16.1.33'
port='1433'
database= 'CUN_REPOSITORIO'

try:
    # Crea la cadena de conexión
   #url = 'mssql+pyodbc://@{host}:{port}/{db}?driver=SQL+Server'.format(host=host, port=port, db=database)
   url = 'mssql+pyodbc://@{host}:{port}/{db}?trusted_connection=yes&driver=SQL+Server'.format(host=host, port=port, db=database)

   
    # Crear conexion con base de datos
   engine = create_engine(url) 
   print("Conexion a la base de datos realizada") 
    
except Exception as e:
    print('Error:',e)

Conexion a la base de datos realizada


SE TRAE EL REGISTRO DE LOS CORREOS QUE NO CUENTAN CON UN INICIO DE SESION


In [3]:
with engine.connect() as connection:
    try:

            query = f"""
                    SELECT distinct Email_institucional AS Correo
    FROM CUN_REPOSITORIO.Google.Usuarios_Workspace
    WHERE Titulo = 'ESTUDIANTE' 
    AND Esta_Suspendido = 0 
    AND Ultimo_Inicio_Sesion = '1970-01-01T00:00:00.000Z' 
    AND Unidad_Organizativa <> '/ESTUDIANTES/APLAZADOS'
        
            """
            df_base_Recibida = pd.read_sql(query, engine)
            # Agregar el resultado a la lista si no está vacío

    finally:
        
        connection.close()

SE TRAEN LOS PERIODOS CON CONVOCATORIA ACTIVA

In [4]:
with engine.connect() as connection:
    try:

            query = f"""
                    SELECT PERIODO FROM CUN.MATRIZ_CALENDARIO
                    WHERE ESTADO_CONVOCATORIA = 'ACTIVO'
        
            """
            PERIODOS_ACTIVOS = pd.read_sql(query, engine)
            # Agregar el resultado a la lista si no está vacío

    finally:
        
        connection.close()

DE LOS CORREOS QUE NO TIENEN REGISTRO SE TRAEN LOS DATOS DE LOS ESTUDIANTES CON MATRICULA ACTIVAY ESTADO DE PAGO PAGO


In [5]:
para_orden = df_base_Recibida['Correo'].unique()
valores_unicos_df1_orden = "', '".join(map(str, para_orden))
valores_unicos_df1_orden = f"'{valores_unicos_df1_orden}'"
periodos_unicos = PERIODOS_ACTIVOS['PERIODO'].unique()
valores_unicos_periodos = "', '".join(map(str, periodos_unicos))
valores_unicos_periodos = f"'{valores_unicos_periodos}'"

In [6]:
with engine.connect() as connection:
    try:

            query = f"""
                    SELECT distinct(DOC_ALUM),CONCAT(NOM_TERCERO,' ',PRI_APELLIDO, ' ',SEG_APELLIDO) AS NOMBRE ,EMAIL_INS AS 'EMAIL INSTITUCIONAL',TEL_CELULAR,TEL_CASA,DIRECCION_CASA,EMAIL_PER  FROM ZOHO.BASE_PERSONAS  
                    WHERE EMAIL_INS IN ({valores_unicos_df1_orden})
                    and estado_pago = 'pago'
                    and est_matriculado = '1-activo'
                    AND PERIODO IN ({valores_unicos_periodos})
        
            """
            df_base_persona = pd.read_sql(query, engine)
            # Agregar el resultado a la lista si no está vacío

    finally:
        
        connection.close()

In [7]:
df_base_persona['TEL_CELULAR'] = df_base_persona['TEL_CELULAR'].fillna(df_base_persona['TEL_CASA'])
df_base_persona

,DOC_ALUM,NOMBRE,EMAIL INSTITUCIONAL,TEL_CELULAR,TEL_CASA,DIRECCION_CASA,EMAIL_PER
0,057319618,VICTORIA PANA,victoria.pana@cun.edu.co,3195513339,3195513339,"Carrera 49 # 75 30, Barranquilla-Atlantico-Col...",victoriapana27@yahoo.com
1,1000003468,ANGIE CAICEDO RUEDA,angie.caicedoru@cun.edu.co,3507787732,3507787732,"Carrera 25 # 21 21, Granada-Meta-Colombia",tatiana.caicedo03@gmail.com
2,1000007217,NICOLÁS GARCÍA,nicolas.garciago@cun.edu.co,3246407664,3246407664,"Carrera 8 A # 30 90, Bogotá-Bogotá- DC-Colombia",gonzslezgarcia85@gmail.com
3,1000017479,RUBEN MORENO BOLAÑOS,ruben.morenob@cun.edu.co,3237932610,3237932610,"Carrera 29 # 10 69, Bogotá-Bogotá- DC-Colombia",ruben10261104@gmail.com
4,1000020216,RAFAEL ACOSTA GUEVARA,rafael.acostag@cun.edu.co,3228387199,3228387199,CL 20 23 21,alejandroacostara2002@gmail.com
...,...,...,...,...,...,...,...
2688,98381879,DIDIER LEIVA GARCIA,didier.leiva@cun.edu.co,3222239104,3222239104,"Carrera 13 A # 10 4, Florencia-Caqueta-Colombia",didierleivag72@gmail.com
2689,98400070,JIMMY PANTOJA MENESES,jimmy.pantojam@cun.edu.co,3192032317,3192032317,bogota,pantojimmy78@gmail.com
2690,98468588,HERNAN SALAZAR SÁNCHEZ,hernan.salazar@cun.edu.co,3113108466,3113108466,"Diagonal 54 # 21 58, Bello-Antioquia-Colombia",alexandersalsan@hotmail.com
2691,98654517,MARIO MADERA MORA,mario.madera@cun.edu.co,3137675499,3137675499,"Calle 10 A # 30 42, Caucasia-Antioquia-Colombia",mmaderam@gmail.com


In [8]:
df_base_persona = df_base_persona.drop_duplicates()
df_base_persona.shape

(2693, 7)

AHORA SE TRAEN LOS ESTUDIANTES SIN REGISTRO EN MOODLE

In [9]:
with engine.connect() as connection:
    try:

            query = f"""
                    SELECT DIR_EMAIL FROM dwh_Inicio_clases
                    WHERE EstadoMoodle = 'Sin registro'
                    and COD_PERIODO IN ({valores_unicos_periodos})
        
            """
            df_moodle = pd.read_sql(query, engine)
            # Agregar el resultado a la lista si no está vacío

    finally:
        
        connection.close()
df_moodle

,DIR_EMAIL
0,juan.ramirezji@cun.edu.co
1,juan.telleze@cun.edu.co
2,mayra.contrerass@cun.edu.co
3,lina.julio@cun.edu.co
4,ender.vidal@cun.edu.co
...,...
2265,hernando.ariass@cun.edu.co
2266,mario.rondon@cun.edu.co
2267,jose.tiques@cun.edu.co
2268,gustavo.velosa@cun.edu.co


In [10]:
para_orden = df_moodle['DIR_EMAIL'].unique()
valores_unicos = "', '".join(map(str, para_orden))
valores_unicos = f"'{valores_unicos}'"

SE TRAEN LOS DATOS DE LOS ESTUDIANTE CON MATRICULA ACTIVA Y ESTADO PAGO = PAGO

In [11]:
with engine.connect() as connection:
    try:

            query = f"""
                    SELECT distinct(DOC_ALUM),CONCAT(NOM_TERCERO,' ',PRI_APELLIDO, ' ',SEG_APELLIDO) AS NOMBRE ,EMAIL_INS AS 'EMAIL INSTITUCIONAL',TEL_CELULAR,TEL_CASA,DIRECCION_CASA,EMAIL_PER  FROM ZOHO.BASE_PERSONAS  
                    WHERE EMAIL_INS IN ({valores_unicos})
                    and estado_pago = 'pago'
                    and est_matriculado = '1-activo'
                    AND PERIODO IN ({valores_unicos_periodos})
        
            """
            df_base_persona_moodle = pd.read_sql(query, engine)
            # Agregar el resultado a la lista si no está vacío

    finally:
        
        connection.close()

In [12]:
#Se asigna el tel de la casa a los vacios de tel_celular
df_base_persona_moodle['TEL_CELULAR'] = df_base_persona_moodle['TEL_CELULAR'].fillna(df_base_persona_moodle['TEL_CASA'])
df_base_persona_moodle.shape

(2250, 7)

In [13]:
#se eliminan duplicados
df_base_persona_moodle = df_base_persona_moodle.drop_duplicates()
df_base_persona_moodle

,DOC_ALUM,NOMBRE,EMAIL INSTITUCIONAL,TEL_CELULAR,TEL_CASA,DIRECCION_CASA,EMAIL_PER
0,1000065820,KAROL MONTES RATIVA,karol.montes@cun.edu.co,3213900783,3213900783,KR 11 NO 65 70,sofiarativa364@gmail.com
1,1000086945,JUAN RANGEL MUÑOZ,juan.rangelm@cun.edu.co,3053200558,3053200558,"Carrera 49 A # 82 74, Medellin-Antioquia-Colombia",juancm1411@gmail.com
2,1000124273,JULIAN GUILLOT RIVEROS,julian.guillot@cun.edu.co,3103337782,3103337782,"Carrera 78 CONJ # 66 17, Bogotá-Bogotá- DC-Col...",juesrigui16@gmail.com
3,057319618,VICTORIA PANA,victoria.pana@cun.edu.co,3195513339,3195513339,"Carrera 49 # 75 30, Barranquilla-Atlantico-Col...",victoriapana27@yahoo.com
4,1000129635,ANDREA VELASCO AGUILERA,andrea.velascoa@cun.edu.co,3203730086,3203730086,-cll 81 c # 17 k 18,asva50714@gmail.com
...,...,...,...,...,...,...,...
2245,94061457,GERSON MARTINEZ SAMBONI,gerson.martinezs@cun.edu.co,3108470587,3108470587,"Calle 7 A # 49 118, Santiago de Cali-Valle Del...",gmmartinez@hotmail.es
2246,94281873,JULIO MARTINEZ SALAZAR,julio.martinezs@cun.edu.co,3206364329,3206364329,CL 70 NO 3N 80,jcmsevilla@gmail.com
2247,94326731,GUSTAVO VELOSA REYES,gustavo.velosa@cun.edu.co,3164911265,3164911265,"Calle 6 C # 6 12, La Dorada-Caldas-Colombia",velolili16@gmail.com
2248,94356953,DIDIER MOLINA LEY,didier.molinal@cun.edu.co,3128660507,3128660507,"Carrera 7 # 10 52, Bogotá-Bogotá- DC-Colombia",molinaleydidier@gmail.com


In [14]:
##from datetime import datetime
##
##hoy = datetime.now().strftime("%d-%m-%Y")
##print(hoy)
##
##with pd.ExcelWriter(f'BASE_ESTUDIANTES_SINREGISTRO{hoy}.xlsx', engine='openpyxl') as writer:
##    df_base_persona.to_excel(writer, sheet_name='SIN REGISTRO', index=False)
##    df_base_persona_moodle.to_excel(writer, sheet_name='SIN REGISTRO MOODLE', index=False)

In [15]:
with engine.connect() as connection:
    try:

            query = f"""
                    SELECT * FROM CUN_REPOSITORIO.AARBOLEDA.est_sinregistro_google
        
            """
            df_historico_google = pd.read_sql(query, engine)
            # Agregar el resultado a la lista si no está vacío

    finally:
        
        connection.close()
df_historico_google.shape        

(7332, 7)

Se quitan los estudiantes que ya se ha enviado informacion en el transcurso de la semana

In [16]:
df_base_persona['DOC_ALUM'] = df_base_persona['DOC_ALUM'].astype(str)
df_historico_google['DOC_ALUM'] = df_historico_google['DOC_ALUM'].astype(str)

In [17]:
df_base_persona_final = df_base_persona.copy()
df_base_persona_final.shape

(2693, 7)

In [18]:
with engine.connect() as connection:
    try:

            query = f"""
                    SELECT * FROM CUN_REPOSITORIO.AARBOLEDA.est_sinregistro_moodle
        
            """
            df_historico_moodle = pd.read_sql(query, engine)
            # Agregar el resultado a la lista si no está vacío

    finally:
        
        connection.close()
df_historico_moodle.shape       

(6100, 7)

In [19]:
df_base_persona_moodle['DOC_ALUM'] = df_base_persona_moodle['DOC_ALUM'].astype(str)
df_historico_moodle['DOC_ALUM'] = df_historico_moodle['DOC_ALUM'].astype(str)

In [20]:
df_base_persona_moodle.shape

(2250, 7)

In [21]:
df_base_persona.shape

(2693, 7)

In [22]:
df_base_persona_moodle_final = df_base_persona_moodle.copy()
df_base_persona_moodle_final.shape

(2250, 7)

In [23]:
from datetime import datetime

hoy = datetime.now().strftime("%d-%m-%Y")
print(hoy)



06-11-2026


In [24]:
#with pd.ExcelWriter(f'BASE_ESTUDIANTES_SINREGISTRO{hoy}.xlsx', engine='openpyxl') as writer:
#    df_base_persona_final.to_excel(writer, sheet_name='SIN REGISTRO', index=False)
#    df_base_persona_moodle_final.to_excel(writer, sheet_name='SIN REGISTRO MOODLE', index=False)

In [25]:
# Connection to database
host = '172.16.1.33'
port = '1433'
database = 'CUN_REPOSITORIO'
# username = 'swarm_cluster'  # Sustituye 'tu_usuario' por tu nombre de usuario real
# password = 'Repo2023*'  # Sustituye 'tu_contraseña' por tu contraseña real
#url = 'mssql+pyodbc://{username}:{password}@{host}:{port}/{db}?driver=ODBC Driver 17 for SQL Server'.format(username=username, password=password, host=host, port=port, db=database)
url = 'mssql+pyodbc://@{host}:{port}/{db}?driver=ODBC Driver 17 for SQL Server'.format(host=host, port=port, db=database)
engine = create_engine(url, fast_executemany=True)

table_name = 'est_sinregistro_moodle'
schema_name = 'AARBOLEDA'
# Send the data to sql server
df_base_persona_moodle_final.to_sql(
    name=table_name,
    schema=schema_name,
    con=engine,
    if_exists='append',
    index=False,
    chunksize = 10000
)

# End connection
engine.dispose()

In [26]:
# Connection to database
host = '172.16.1.33'
port = '1433'
database = 'CUN_REPOSITORIO'
# username = 'swarm_cluster'  # Sustituye 'tu_usuario' por tu nombre de usuario real
# password = 'Repo2023*'  # Sustituye 'tu_contraseña' por tu contraseña real
#url = 'mssql+pyodbc://{username}:{password}@{host}:{port}/{db}?driver=ODBC Driver 17 for SQL Server'.format(username=username, password=password, host=host, port=port, db=database)
url = 'mssql+pyodbc://@{host}:{port}/{db}?driver=ODBC Driver 17 for SQL Server'.format(host=host, port=port, db=database)
engine = create_engine(url, fast_executemany=True)

table_name = 'est_sinregistro_google'
schema_name = 'AARBOLEDA'
# Send the data to sql server
df_base_persona_final.to_sql(
    name=table_name,
    schema=schema_name,
    con=engine,
    if_exists='append',
    index=False,
    chunksize = 10000
)

# End connection
engine.dispose()

# SE SUBE A LA TABLA QUE SE VA A LEER DIARIAMENTE PARA ENVIAR LA INFORMACION

In [27]:
# Connection to database
host = '172.16.1.33'
port = '1433'
database = 'CUN_REPOSITORIO'
# username = 'swarm_cluster'  # Sustituye 'tu_usuario' por tu nombre de usuario real
# password = 'Repo2023*'  # Sustituye 'tu_contraseña' por tu contraseña real
#url = 'mssql+pyodbc://{username}:{password}@{host}:{port}/{db}?driver=ODBC Driver 17 for SQL Server'.format(username=username, password=password, host=host, port=port, db=database)
url = 'mssql+pyodbc://@{host}:{port}/{db}?driver=ODBC Driver 17 for SQL Server'.format(host=host, port=port, db=database)
engine = create_engine(url, fast_executemany=True)

table_name = 'est_sinregistro_google'
schema_name = 'MOODLE'
# Send the data to sql server
df_base_persona_final.to_sql(
    name=table_name,
    schema=schema_name,
    con=engine,
    if_exists='replace',
    index=False,
    chunksize = 10000
)

# End connection
engine.dispose()

In [28]:
# Connection to database
host = '172.16.1.33'
port = '1433'
database = 'CUN_REPOSITORIO'
# username = 'swarm_cluster'  # Sustituye 'tu_usuario' por tu nombre de usuario real
# password = 'Repo2023*'  # Sustituye 'tu_contraseña' por tu contraseña real
#url = 'mssql+pyodbc://{username}:{password}@{host}:{port}/{db}?driver=ODBC Driver 17 for SQL Server'.format(username=username, password=password, host=host, port=port, db=database)
url = 'mssql+pyodbc://@{host}:{port}/{db}?driver=ODBC Driver 17 for SQL Server'.format(host=host, port=port, db=database)
engine = create_engine(url, fast_executemany=True)

table_name = 'est_sinregistro_moodle'
schema_name = 'MOODLE'
# Send the data to sql server
df_base_persona_moodle_final.to_sql(
    name=table_name,
    schema=schema_name,
    con=engine,
    if_exists='replace',
    index=False,
    chunksize = 10000
)

# End connection
engine.dispose()